# Caso 2 — Modelo Vetorial (Vector Space Model)

Este notebook implementa o **Modelo Vetorial** clássico: cada documento e
cada consulta são representados como vetores no espaço de termos, com peso
TF-IDF, e o ranking é dado pela similaridade de cosseno entre o vetor da
consulta e os vetores dos documentos.


In [1]:
from pathlib import Path
import sys
project_root = Path.cwd()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from src.cranfield_data import load_cranfield
from src.pre_processing import load_preprocessed
from src.vector_model import VectorSpaceModel

df_docs, df_queries, df_qrels = load_cranfield()
preprocessed = load_preprocessed(project_root / "data" / "processed" / "preprocessed_cranfield.pkl")

doc_ids = df_docs["doc_id"].tolist()
query_ids = df_queries["query_id"].tolist()
doc_title = dict(zip(df_docs.doc_id, df_docs.title.str.replace("\n", " ")))


## Configuração de pré-processamento adotada

A partir deste ponto, fixamos uma única configuração de pré-processamento
para permitir comparações justas entre os modelos nos casos seguintes.


In [ ]:
# Configuração de pré-processamento fixa com stopwords e stemming.
#
# O Caso 1 testou 4 configurações de pré-processamento. Esta foi escolhida
# como padrão para os demais notebooks (Vetorial e BM25) porque reduz o
# vocabulário de 6813 para 4117 termos e remove ruído como stopwords e
# variações morfológicas. Assim a comparação entre modelos nos Casos 5 a 9
# usa sempre o mesmo pré-processamento.
CONFIG = "stopwords_stemming"

doc_tokens = preprocessed[CONFIG]["docs"]
query_tokens_list = preprocessed[CONFIG]["queries"]

print(f"Configuração usada: {CONFIG}")
print(f"Documentos: {len(doc_tokens)} | Consultas: {len(query_tokens_list)}")


Configuração usada: stopwords_stemming
Documentos: 1400 | Consultas: 225


## Construção do modelo

In [ ]:
# Construção do Modelo Vetorial
#
# Usa TfidfVectorizer do scikit-learn sobre os documentos já tokenizados
# na configuração de pré-processamento escolhida acima. O peso de cada
# termo é o produto entre a contagem bruta no documento (tf) e o idf com
# smoothing padrão do sklearn. Cada vetor de documento é normalizado em
# L2, também como padrão do sklearn.
#
# A similaridade entre consulta e documento é o cosseno, calculado com
# cosine_similarity. Como os vetores já estão normalizados em L2, isso
# equivale ao produto escalar entre eles. A classe VectorSpaceModel em
# src/vector_model.py encapsula essa construção para que o mesmo espaço
# vetorial, com vocabulário e idf ajustados sobre os documentos, seja
# reutilizado ao rankear cada consulta.
vsm = VectorSpaceModel(doc_tokens)
print(f"Vocabulário do modelo vetorial: {len(vsm.vocab)} termos")
print(f"Matriz documento-termo: {vsm.doc_matrix.shape}, "
      f"{vsm.doc_matrix.nnz} entradas não-nulas "
      f"({100*vsm.doc_matrix.nnz/(vsm.doc_matrix.shape[0]*vsm.doc_matrix.shape[1]):.2f}% de densidade)")


Vocabulário do modelo vetorial: 4116 termos
Matriz documento-termo: (1400, 4116), 79339 entradas não-nulas (1.38% de densidade)


## Exemplo de recuperação

Abaixo, o Top-10 de documentos recuperados pelo Modelo Vetorial para a
primeira consulta da coleção, indicando o grau de relevância (segundo o
qrels) de cada documento retornado.


In [4]:
# Exemplo: ranking para a primeira consulta da coleção
query_id_0 = query_ids[0]
print(f"Consulta {query_id_0}: {df_queries.iloc[0]['text']}")
print(f"Tokens (pós pré-processamento): {query_tokens_list[0]}")
print()

ranking = vsm.rank(query_tokens_list[0], top_n=10)
print("Top-10 documentos recuperados pelo Modelo Vetorial:")
rows = []
grades = dict(zip(df_qrels[df_qrels.query_id == query_id_0].doc_id,
                   df_qrels[df_qrels.query_id == query_id_0].relevance))
for rank, (doc_idx, score) in enumerate(ranking, start=1):
    did = doc_ids[doc_idx]
    grade = grades.get(did, None)
    rows.append({
        "rank": rank, "doc_id": did, "score_cosseno": round(score, 4),
        "grau_relevancia": grade if grade is not None else "não julgado",
        "titulo": doc_title[did][:80],
    })
display(pd.DataFrame(rows))


Consulta 1: what similarity laws must be obeyed when constructing aeroelastic models
of heated high speed aircraft .
Tokens (pós pré-processamento): ['similar', 'law', 'must', 'obey', 'construct', 'aeroelast', 'model', 'heat', 'high', 'speed', 'aircraft']

Top-10 documentos recuperados pelo Modelo Vetorial:


,rank,doc_id,score_cosseno,grau_relevancia,titulo
0,1,51,0.3123,3,theory of aircraft structural models subjected...
1,2,746,0.2481,não julgado,aeroelastic problems in connection with high s...
2,3,359,0.2249,não julgado,note on the hypersonic similarity law for an u...
3,4,13,0.2073,4,similarity laws for stressing heated wings .
4,5,875,0.2015,2,models for aeroelastic investigation .
5,6,12,0.1972,3,some structural and aerelastic considerations ...
6,7,486,0.1957,-1,similarity laws for aerothermoelastic testing .
7,8,879,0.1840,3,flutter model testing at transonic speeds .
8,9,56,0.1812,3,an analysis of the applicability of the hypers...
9,10,874,0.1811,não julgado,the use of models for the determination of cri...


## Verificação de sanidade do IDF

Termos raros (que aparecem em poucos documentos) devem ter idf alto e
pesar mais nas comparações; termos muito comuns devem ter idf baixo. Isso
confirma que a ponderação está calculada corretamente antes de seguirmos
para o BM25 e a avaliação quantitativa.


In [5]:
# Verificação de sanidade: a consulta é mais similar a si mesma (como
# "pseudo-documento") do que documentos aleatórios são entre si, e termos
# muito frequentes na coleção (baixo idf) pesam pouco no ranking.
top_idf_terms = sorted(zip(vsm.vocab, vsm.idf), key=lambda x: -x[1])[:5]
bottom_idf_terms = sorted(zip(vsm.vocab, vsm.idf), key=lambda x: x[1])[:5]
print("Termos mais raros (maior idf) — carregam mais peso no cosseno:")
for t, v in top_idf_terms:
    print(f"  {t}: idf={v:.3f}")
print("\nTermos mais comuns (menor idf) — carregam menos peso:")
for t, v in bottom_idf_terms:
    print(f"  {t}: idf={v:.3f}")


Termos mais raros (maior idf) — carregam mais peso no cosseno:
  ab: idf=7.552
  abbrevi: idf=7.552
  abrupt: idf=7.552
  absent: idf=7.552
  abundantli: idf=7.552

Termos mais comuns (menor idf) — carregam menos peso:
  flow: idf=1.674
  result: idf=1.707
  number: idf=1.901
  effect: idf=1.959
  pressur: idf=1.966
